In [ ]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT_DIR = Path.cwd()
ss = ROOT_DIR/'data'/'superstoreSales.csv'
df = pd.read_csv(ss, encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Order Priority,Order Quantity,Sales,Discount,Ship Mode,Profit,Unit Price,...,Customer Name,Province,Region,Customer Segment,Product Category,Product Sub-Category,Product Name,Product Container,Product Base Margin,Ship Date
0,1,3,10/13/2010,Low,6,261.5400,0.04,Regular Air,-213.25,38.94,...,Muhammed MacIntyre,Nunavut,Nunavut,Small Business,Office Supplies,Storage & Organization,"Eldon Base for stackable storage shelf, platinum",Large Box,0.80,10/20/2010
1,49,293,10/1/2012,High,49,10123.0200,0.07,Delivery Truck,457.81,208.16,...,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Jumbo Drum,0.58,10/2/2012
2,50,293,10/1/2012,High,27,244.5700,0.01,Regular Air,46.71,8.69,...,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Binders and Binder Accessories,"Cardinal Slant-Dï¿½ Ring Binder, Heavy Gauge V...",Small Box,0.39,10/3/2012
3,80,483,7/10/2011,High,30,4965.7595,0.08,Regular Air,1198.97,195.99,...,Clay Rozendal,Nunavut,Nunavut,Corporate,Technology,Telephones and Communication,R380,Small Box,0.58,7/12/2011
4,85,515,8/28/2010,Not Specified,19,394.2700,0.08,Regular Air,30.94,21.78,...,Carlos Soltero,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,Holmes HEPA Air Purifier,Medium Box,0.50,8/30/2010


In [18]:
sum_df = pd.DataFrame({
    'missing': df.isnull().sum(),
    'qunique': df.nunique(),
    'data_type': df.dtypes
})

sum_df

,missing,qunique,data_type
Row ID,0,232,int64
Order ID,0,159,int64
Order Date,0,153,datetime64[us]
Order Priority,0,5,str
Order Quantity,0,50,int64
Sales,0,232,float64
Discount,0,12,float64
Ship Mode,0,3,str
Profit,0,232,float64
Unit Price,0,177,float64


Convert Order Date to datetime, set it as the index, and resample to monthly total
sales.

In [9]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df = df.set_index(df['Order Date'])

In [16]:
monthly = df.groupby(df['Order Date'].dt.month)['Sales'].sum()
monthly

Order Date
1     70114.5350
2     40092.4340
3     25187.7740
4     16425.9170
5     14697.7560
6     60681.4885
7     13505.4395
8     67777.6100
9     24638.4710
10    70939.2705
11    37524.3040
12    29693.6385
Name: Sales, dtype: float64

Add a 3-month moving average of monthly sales.

In [19]:
monthly.rolling(3).mean()

Order Date
1              NaN
2              NaN
3     45131.581000
4     27235.375000
5     18770.482333
6     30601.720500
7     29628.228000
8     47321.512667
9     35307.173500
10    54451.783833
11    44367.348500
12    46052.404333
Name: Sales, dtype: float64

Compute month-over-month change in sales using shift .

In [21]:
monthly_change = monthly - monthly.shift(1)
monthly_change

Order Date
1            NaN
2    -30022.1010
3    -14904.6600
4     -8761.8570
5     -1728.1610
6     45983.7325
7    -47176.0490
8     54272.1705
9    -43139.1390
10    46300.7995
11   -33414.9665
12    -7830.6655
Name: Sales, dtype: float64

Group by weekday name to see which day of the week has the highest average order
Sales .

In [30]:
sales_day = df.groupby(df['Order Date'].dt.day_name())['Sales'].mean()
sales_day.sort_values(ascending=False)

Order Date
Tuesday      2847.116654
Sunday       2262.065552
Thursday     2240.484286
Wednesday    1746.444857
Monday       1746.198806
Friday       1672.050933
Saturday     1539.544742
Name: Sales, dtype: float64